# Lesson 03: LLM API Quickstart & Prompt Patterns

### How it works:
1. **API Key:** Get your free key at [Google AI Studio](https://aistudio.google.com/) ($0 cost, no credit card required).
2. **Colab Secrets:** Click the **Key icon (Secrets)** on the left toolbar, add name `GEMINI_API_KEY`, paste your key, and enable **Notebook access**.
3. **Happy Flow:** Connect, call the model, inspect tokens, and test prompt patterns.

In [1]:
# 1. Install Google GenAI library (~5 seconds)
!pip install -q -U google-genai

# 2. Connect to the API using your secret key
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Connected to Gemini API!")

Connected to Gemini API!


---
## 1. First API Call & Response
Send a simple question to `gemini-3.5-flash-lite` and print the response.

In [2]:
response = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents='What is an API in one simple sentence?'
)

print(response.text.strip())

An API is a software messenger that allows different applications to communicate and exchange data automatically.


---
## 2. Token Telemetry (Digital Receipt)
Check how many tokens were used for input and output.

In [3]:
print(f"Prompt tokens:    {response.usage_metadata.prompt_token_count}")
print(f"Candidate tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:     {response.usage_metadata.total_token_count}")

Prompt tokens:    11
Candidate tokens: 19
Total tokens:     30


---
## 3. Prompt Patterns on a Real Task
We now analyze a real customer support message using the prompt patterns from Lesson 02.

In [4]:
ticket = "I ordered 2 SSD drives (Order #48192). Only 1 arrived. Refund $90 please."
print("Ticket:", ticket)

Ticket: I ordered 2 SSD drives (Order #48192). Only 1 arrived. Refund $90 please.


### Pattern A: Zero-Shot (Instructions Only)

In [5]:
prompt_zero = f"Extract order_id and refund_usd into JSON from this ticket:\n{ticket}"

resp_zero = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_zero
)
print(resp_zero.text.strip())

{
  "order_id": "48192",
  "refund_usd": 90.0
}


### Pattern B: One-Shot (1 Demonstration Example)
Adding 1 example guides the model to follow the exact format.

In [6]:
prompt_one = f"""Extract order_id and refund_usd into JSON.

Example:
Ticket: "Order #111 missing 1 cable. Refund $15."
JSON: {{"order_id": "111", "refund_usd": 15}}

Ticket:
{ticket}"""

resp_one = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_one
)
print(resp_one.text.strip())

{
  "order_id": "48192",
  "refund_usd": 90.0
}


### Pattern C: Few-Shot (Multiple Examples)
Adding a second example handles edge cases (e.g. no refund needed).

In [7]:
prompt_few = f"""Extract order_id and refund_usd into JSON.

Example 1:
Ticket: "Order #111 missing 1 cable. Refund $15."
JSON: {{"order_id": "111", "refund_usd": 15}}

Example 2:
Ticket: "Order #222 is delayed. No refund needed."
JSON: {{"order_id": "222", "refund_usd": 0}}

Ticket:
{ticket}"""

resp_few = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_few
)
print(resp_few.text.strip())

{
  "order_id": "48192",
  "refund_usd": 90.0
}


### Pattern D: XML Structure (Production Standard)
Separating rules, examples, and user input with XML tags ensures 100% parseable JSON.

In [8]:
prompt_xml = f"""<rules>
Extract order_id and refund_usd into valid JSON.
</rules>

<examples>
<example>
<ticket>Order #111 missing 1 cable. Refund $15.</ticket>
<json>{{"order_id": "111", "refund_usd": 15}}</json>
</example>
</examples>

<ticket>
{ticket}
</ticket>"""

resp_xml = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_xml
)
print(resp_xml.text.strip())

{
  "order_id": "48192",
  "refund_usd": 90.0
}


---
## 4. Student Playground
Modify `my_ticket` below and run the cell to test extraction on your own message.

In [9]:
# Type your own ticket message:
my_ticket = "Hello, order #99231 arrived broken. Need $45 refund."

my_prompt = f"""<rules>
Extract order_id and refund_usd into valid JSON.
</rules>
<ticket>
{my_ticket}
</ticket>"""

my_response = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=my_prompt
)
print(my_response.text.strip())

{
  "order_id": "99231",
  "refund_usd": 45.0
}
